TEST


In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import warnings
warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

sys.path.append('/home/adelval/BTS/TFM/test/src/net')
sys.path.append('/home/adelval/BTS/TFM/test/src/train')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration

In [2]:
import os
import numpy as np
import gzip

def read_list_str(file, list_=None):
    if isinstance(file,list):
        list_ = []
        for filei in file:
           list_ = read_list_str(filei, list_=list_)
        return list_
    
    if isinstance(file,str):   
        root,ext = os.path.splitext(file)
        if ext == '.gz':
            with gzip.open(file, 'r') as f:
                return read_list_str(f, list_=list_)
        else:
            with open(file, 'r') as f:
                return read_list_str(f, list_=list_)
    else:
        if list_ is None:
            list_ = []
            
        for line in file.readlines():
            list_.append( line.strip() )
    return list_

def read_list_str_np(file):    
    return np.array(read_list_str(file))


def read_table_str(file, table=None):
    if isinstance(file,list):
        table = []
        for filei in file:
           table = read_table_str(filei, table=table)
        return table
    
    if isinstance(file,str):   
        root,ext = os.path.splitext(file)
        if ext == '.gz':
            with gzip.open(file, 'r') as f:
                return read_table_str(f, table=table)
        else:
            with open(file, 'r') as f:
                return read_table_str(f, table=table)
    else:
        if table is None:
            table = []
            
        for line in file.readlines():
            table.append( line.split() )
    return table

def read_table_str_np(file):    
    return np.array(read_table_str(file))

In [3]:
file_ = []

file_list_ = ["/home/adelval/BTS/TFM/test/data/lst/minitest_16k.wav.list"]

for file_list in file_list_:
    file_ += read_list_str(file_list)

x_test = file_


print('  x_test: %s' % len(x_test))
print(x_test)

  x_test: 9
['data/audio/minitest_16k/10-CH0_C01_airport_10dB.wav', 'data/audio/minitest_16k/11-CH0_C01_construction_5dB.wav', 'data/audio/minitest_16k/12-CH0_C01_babies_5dB.wav', 'data/audio/minitest_16k/14-CH0_C01_callcenters_5dB.wav', 'data/audio/minitest_16k/15-CH0_C01_barks_5dB.wav', 'data/audio/minitest_16k/16-CH0_C01_barks_5dB.wav', 'data/audio/minitest_16k/5-CH0_C01_stadium_15dB.wav', 'data/audio/minitest_16k/6-CH0_C01_traffic_15dB.wav', 'data/audio/minitest_16k/7-CH0_C01_city_5dB.wav']


In [4]:
x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
print('  x_test: %s' % len(x_test))
print(x_test)

  x_test: 1
['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [5]:
from collections import OrderedDict

data = ('file', x_test)
fieldx = data[0]
data = data[1]
index = 0
data_i = OrderedDict()

data_i[fieldx] = data[index]    
print(data_i)

OrderedDict([('file', '/home/adelval/BTS/TFM/audios/audio_1.wav')])


In [6]:
#ReadAudio

import soundfile as sf
def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

def get_key_path(file_list=None, dir=None, ext=None):
    if file_list is not None:        
        if isinstance(file_list, str):
            file_list = read_table_str_np(file_list)
        file_list = {key: path for key, path in file_list}
        class get_key_path_from_list:
            def __init__(self, file_list):
                self.file_list = file_list            
            def __call__(self, key):
                return self.file_list[key]
        return get_key_path_from_list(file_list)
        
    else:
        if dir is None:
            return lambda key: key+ext
        else:
            return lambda key: dir+'/'+key+ext
        
def copy_array_dict(d,x,k):
    if isinstance(k, list) or isinstance(k, tuple):
        if len(k) > 1:
            for ki in k:
                d[ki] = x.copy()
        else:
            d[k[0]] = x
    else:
        d[k] = x

var = 'x08k'
var_input = 'file'
key2path = get_key_path(ext='')
data_i['file'] = '/home/adelval/BTS/TFM/audios/audio_1.wav'
print(data_i)
audio = key2path(data_i[var_input])
x, data_i['fs'] = read_audio(audio)
copy_array_dict(data_i, x, var)
print(data_i)

data_i['x08k2'] = data_i[var].copy()
print(data_i['x08k'])


OrderedDict([('file', '/home/adelval/BTS/TFM/audios/audio_1.wav')])
OrderedDict([('file', '/home/adelval/BTS/TFM/audios/audio_1.wav'), ('fs', 16000), ('x08k', array([  0,  -1,  -1, ..., -30, -34, -30], dtype=int16))])
[  0  -1  -1 ... -30 -34 -30]


In [7]:
#FFT
from spicy import signal

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010):
    N = int(Ns * fs)            # Number of samples in each window
    M = int(Ms * fs)            # Step size (number of samples between window starts)
    n = (len(x) + M - 1) // M   # Number of frames
    print("Number of frames", n)    
    T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    if T > len(x):
        xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, n * M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    xa = xa[ind.astype(int).T].astype(np.float32)
    # print(f'0 Window frames {xa[:20,:10]}')
    return xa

def hamming(X):
    print("la de X ", X.shape)
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[1])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=1)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:, :(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals

field='x08k'
fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]

N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
#fb = [fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]

x = data_i[field]
print(f'0 frame   {x[:10]}')
print(f'1 frame   {x[1*int(m*fs):1*int(m*fs)+10]}')
print(f'2 frame   {x[2*int(m*fs):2*int(m*fs)+10]}')
print(f'3 frame   {x[3*int(m*fs):3*int(m*fs)+10]}')
print(f'23 frame  {x[22*int(m*fs):22*int(m*fs)+10]}')
#x = x / (np.abs(x).max()+1e-8) * .95 # probar a quitar porque si tenemos una señal cero lo puede amplificar a 
# To remove DC offset
x = offset(x)
# Emphasis to increase the amplitude of high freq
x = preemphasis(x)
print(f'0 frame   {x[:10]}')
print(f'1 frame   {x[1*int(m*fs):1*int(m*fs)+10]}')
print(f'2 frame   {x[2*int(m*fs):2*int(m*fs)+10]}')
print(f'3 frame   {x[3*int(m*fs):3*int(m*fs)+10]}')
print(f'23 frame  {x[22*int(m*fs):22*int(m*fs)+10]}')
XX = []
for i,w in enumerate(w):
    X = hamming(windowing2(x, fs=fs, Ns=w, Ms=m))
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[i])
    print(f"la shape de Xfft es {Xfft.shape}")
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)

data_i[field] = X
print("0 Vector 2D con FFT para cada frame por row \n", data_i['x08k'][0,:10])
print("1 Vector 2D con FFT para cada frame por row \n", data_i['x08k'][1,:10])
print("2 Vector 2D con FFT para cada frame por row \n", data_i['x08k'][2,:10])
print("3 Vector 2D con FFT para cada frame por row \n", data_i['x08k'][3,:10])
print("20 Vector 2D con FFT para cada frame por row \n", data_i['x08k'][19,:10])


0 frame   [ 0 -1 -1  0 -1  0  0  0  1  1]
1 frame   [-1  0  0  0  0  0  0 -1 -1  0]
2 frame   [ 0  0  0  0  0  1  0 -1 -1  0]
3 frame   [ 0  0 -1  0  0  0  0  0  0  1]
23 frame  [ 1  0  1  0  0  0  1  0 -1  0]
0 frame   [ 0.00000000e+00 -1.00000000e+00 -2.89900000e-02  9.71039280e-01
 -9.99941470e-01  9.71068471e-01  8.76919559e-05  8.76033871e-05
  1.00008751e+00  2.90774265e-02]
1 frame   [-1.00007831e+00  9.70931768e-01 -4.88727982e-05 -4.88234367e-05
 -4.87741250e-05 -4.87248632e-05 -4.86756510e-05 -1.00004863e+00
 -2.90385774e-02  9.70990752e-01]
2 frame   [-9.70854401e-01  1.26161699e-04  1.26034276e-04  1.25906981e-04
  1.25779815e-04  1.00012565e+00 -9.70884474e-01 -9.99903881e-01
 -2.88939779e-02  9.71135205e-01]
3 frame   [-4.07759293e-05 -4.07347456e-05 -1.00004069e+00  9.70969347e-01
 -1.13315439e-05 -1.13200991e-05 -1.13086658e-05 -1.12972440e-05
 -1.12858338e-05  9.99988726e-01]
23 frame  [ 1.00000433e+00 -9.71005672e-01  9.99975043e-01 -9.71034931e-01
 -5.41862177e-05 -5

In [8]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients

def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b



field='x08k2'
fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
dct = [ f_base_dct(Bi) for Bi in B] 


x = data_i[field]
x = offset(x)
x = preemphasis(x)
XX = []
for i,w in enumerate(w):
    X = hamming(windowing2(x, fs=fs, Ns=w, Ms=m))
    
    Xfft = fft(X, nfft[i])
    Xb = np.log(Xfft.dot( fb[i] ) + 1)
    Xc = Xb.dot(dct[i])                                
    
    X = np.concatenate( [Xb, Xc], 1 )
    
    X = np.asarray(X, dtype=np.float32)
    print(X.shape)
    print(f'0 Filtro Mel {X[0,:5]}')
    print(f'1 Filtro Mel {X[1,:5]}')
    print(f'2 Filtro Mel {X[2,:5]}')
    print(f'3 Filtro Mel {X[3,:5]}')
    print(f'19 Filtro Mel {X[19,:5]}')
    XX.append(X)
XX = np.concatenate(XX, 1)
  

data_i[field] = XX
print("El tamaño de x08k2 es: ",data_i[field].shape)
print("Vector 2D con FFT para cada frame por row \n", data_i['x08k2'][0,:])

Number of frames 466
la de X  (466, 640)
(466, 64)
0 Filtro Mel [0.23243642 0.28519294 0.5673354  0.683461   0.7875668 ]
1 Filtro Mel [0.21219873 0.25336957 0.5756598  0.7295447  0.95927125]
2 Filtro Mel [0.21223295 0.25848418 0.41061014 0.50089383 0.77805275]
3 Filtro Mel [0.34777853 0.36755627 0.45605582 0.41130796 0.7351749 ]
19 Filtro Mel [0.34695724 0.57377666 0.37493095 0.4249149  0.5653557 ]
El tamaño de x08k2 es:  (466, 64)
Vector 2D con FFT para cada frame por row 
 [ 2.3243642e-01  2.8519294e-01  5.6733543e-01  6.8346101e-01
  7.8756678e-01  7.0884508e-01  5.9491795e-01  8.4041607e-01
  1.4721725e+00  1.1784921e+00  9.0479809e-01  1.6591580e+00
  1.6208335e+00  1.6077632e+00  1.6622918e+00  1.8071356e+00
  1.8437001e+00  2.0725408e+00  1.9555092e+00  2.0416002e+00
  2.2954106e+00  2.2919426e+00  2.3828988e+00  2.2032447e+00
  2.2712402e+00  2.2792289e+00  2.4267817e+00  2.4498546e+00
  2.5948927e+00  2.6776021e+00  2.5291076e+00  2.5170765e+00
  9.4479094e+00 -4.0548253e+00 -

In [9]:
# Log scale
scale=1.
eps=1e-8
var = 'x08k'
x = data_i[var]     
x = np.abs(x)
x = scale * np.log10(x + eps)      

print(data_i['x08k'][0,:10])

data_i[var] = x   
print("El tamaño de x08k en log es: ",data_i[var].shape)
print(data_i['x08k'][0,:10])
print(data_i['x08k'][1,:10])
print(data_i['x08k'][2,:10])
print(data_i['x08k'][3,:10])
print(data_i['x08k'][19,:10])

[0.00335845 0.03019467 0.0889347  0.14395647 0.19585675 0.20879966
 0.09916716 0.03259555 0.05726052 0.07491741]
El tamaño de x08k en log es:  (466, 512)
[-2.47386    -1.5200697  -1.0509288  -0.8417688  -0.70806146 -0.6802702
 -1.0036321  -1.4868416  -1.2421447  -1.1254172 ]
[-1.5796448  -1.5199704  -1.7905464  -1.2009634  -0.7062188  -0.5960009
 -0.86695796 -1.4692198  -1.3591225  -1.2687083 ]
[-1.3838824 -1.1336191 -1.7873496 -2.563182  -1.5647051 -0.5705921
 -0.7878806 -1.470212  -1.2014716 -2.1963046]
[-1.0911639  -1.4584094  -1.1114112  -0.90263665 -0.8511954  -1.2776407
 -0.66368127 -0.50070375 -0.6474159  -0.7516225 ]
[-1.3306018  -1.2445422  -1.3499471  -1.991468   -3.9184818  -1.872587
 -1.0929663  -0.6899848  -0.78880394 -1.5080434 ]


In [10]:
# Pasa a escala log x08k, es decir, la pds de los frames
def return_dict_values(d):
    v = list( d.values() )
    if len(v) == 1:
        return v[0]
    else:
        return v
    
print(return_dict_values(data_i))

['/home/adelval/BTS/TFM/audios/audio_1.wav', 16000, array([[-2.47386   , -1.5200697 , -1.0509288 , ...,  2.9450564 ,
         2.5842927 ,  2.1603403 ],
       [-1.5796448 , -1.5199704 , -1.7905464 , ...,  2.3097441 ,
         2.2818258 ,  2.3982136 ],
       [-1.3838824 , -1.1336191 , -1.7873496 , ...,  2.4780755 ,
         2.336498  ,  2.082412  ],
       ...,
       [ 2.6044483 ,  2.825645  ,  3.0103223 , ...,  2.575078  ,
         2.3680224 ,  1.6045561 ],
       [ 2.8390841 ,  2.8450115 ,  2.8543947 , ...,  2.073546  ,
         1.9089843 ,  1.5054749 ],
       [ 1.5995234 ,  1.6007298 ,  1.6042771 , ...,  0.24609901,
         0.17873281,  0.12766637]], dtype=float32), array([[ 0.23243642,  0.28519294,  0.5673354 , ...,  0.13855568,
        -0.08115076, -0.03467707],
       [ 0.21219873,  0.25336957,  0.5756598 , ...,  0.11903407,
         0.07097483,  0.00531933],
       [ 0.21223295,  0.25848418,  0.41061014, ..., -0.18487273,
         0.03378217,  0.02448292],
       ...,
       

In [11]:
import pickle

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

file = '/home/adelval/BTS/TFM/test/data/model/fe1_norm1.pkl'  # de donde salen??
mu, std = read_pkl(file)
x = data_i['x08k2']  
print(np.mean(data_i['x08k2']))
x -= mu
x /= std + 1e-6
data_i['x08k2'] = x 

print("El tamaño de x08k2 en log es: ",data_i['x08k2'].shape)
print(np.mean(data_i['x08k2']))
print((data_i['x08k2'][0,:5]))
print((data_i['x08k2'][1,:5]))
print((data_i['x08k2'][2,:5]))
print((data_i['x08k2'][3,:5]))
print((data_i['x08k2'][19,:5]))

3.3606358
El tamaño de x08k2 en log es:  (466, 64)
-0.23491597
[-2.984838  -2.8981009 -2.7997372 -2.7685244 -2.7185802]
[-2.9946365 -2.9120762 -2.796107  -2.7484264 -2.6457765]
[-2.9946198 -2.90983   -2.8680854 -2.8481464 -2.7226143]
[-2.9289918 -2.861931  -2.8482666 -2.8872168 -2.740795 ]
[-2.9293892 -2.771369  -2.8836453 -2.8812828 -2.8127995]


In [12]:
# Concatenación de x08k y x08k2
data_ = []
var = ('x08k','x08k2')
var1 = 'x08k'
for var in var:
    data_.append(data_i[var])

 
data_i[var1] = np.concatenate( data_, 1 )
print("El tamaño de x08k tras la concatenacion es: ",data_i['x08k'].shape)
print(f'0 La concatenacion resulta {data_i["x08k"][0,:5]} y {data_i["x08k"][0,512:517]}')
print(f'1 La concatenacion resulta {data_i["x08k"][1,:5]} y {data_i["x08k"][1,512:517]}')
print(f'2 La concatenacion resulta {data_i["x08k"][2,:5]} y {data_i["x08k"][2,512:517]}')
print(f'3 La concatenacion resulta {data_i["x08k"][3,:5]} y {data_i["x08k"][3,512:517]}')
print(f'19 La concatenacion resulta {data_i["x08k"][19,:5]} y {data_i["x08k"][19,512:517]}')


El tamaño de x08k tras la concatenacion es:  (466, 576)
0 La concatenacion resulta [-2.47386    -1.5200697  -1.0509288  -0.8417688  -0.70806146] y [-2.984838  -2.8981009 -2.7997372 -2.7685244 -2.7185802]
1 La concatenacion resulta [-1.5796448 -1.5199704 -1.7905464 -1.2009634 -0.7062188] y [-2.9946365 -2.9120762 -2.796107  -2.7484264 -2.6457765]
2 La concatenacion resulta [-1.3838824 -1.1336191 -1.7873496 -2.563182  -1.5647051] y [-2.9946198 -2.90983   -2.8680854 -2.8481464 -2.7226143]
3 La concatenacion resulta [-1.0911639  -1.4584094  -1.1114112  -0.90263665 -0.8511954 ] y [-2.9289918 -2.861931  -2.8482666 -2.8872168 -2.740795 ]
19 La concatenacion resulta [-1.3306018 -1.2445422 -1.3499471 -1.991468  -3.9184818] y [-2.9293892 -2.771369  -2.8836453 -2.8812828 -2.8127995]


In [13]:
# Delete dict parts except file and x08k
var2 = ('file','x08k')
for var in list(data_i.keys()):
    if not var in var2:
        data_i.pop(var, None)   
print(data_i.keys())    
print(data_i.values())

odict_keys(['file', 'x08k'])
odict_values(['/home/adelval/BTS/TFM/audios/audio_1.wav', array([[-2.47386   , -1.5200697 , -1.0509288 , ...,  0.44455567,
        -0.27737492, -0.1630669 ],
       [-1.5796448 , -1.5199704 , -1.7905464 , ...,  0.37889755,
         0.24361184,  0.00419377],
       [-1.3838824 , -1.1336191 , -1.7873496 , ..., -0.64324915,
         0.11623757,  0.08433389],
       ...,
       [ 2.6044483 ,  2.825645  ,  3.0103223 , ...,  0.02514895,
         0.47776192, -0.6993419 ],
       [ 2.8390841 ,  2.8450115 ,  2.8543947 , ...,  0.2836317 ,
         0.51829845,  0.2287683 ],
       [ 1.5995234 ,  1.6007298 ,  1.6042771 , ...,  0.69107956,
        -0.06538004,  0.19398205]], dtype=float32)])


In [14]:
import torch 
from torch.autograd import Variable

def to_variable(var=(), cuda=True, float16=False, volatile=False):
    out = []
    for v in var:
        if isinstance(v, np.ndarray):
            v = torch.from_numpy(v)
        
        if float16 and v.dtype==torch.float32:
            v = v.half()

        if not v.is_cuda and cuda:
            v = v.cuda()

        if not isinstance(v, Variable):
            v = Variable(v, volatile=volatile)
        out.append(v)
    return out

x = data_i['x08k']
print(x.shape)
x,  = to_variable(var=(x, ), volatile=True, cuda=False, float16=False)
print(x.shape)

(466, 576)
torch.Size([466, 576])


In [15]:
import pickle
def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj('/home/adelval/BTS/TFM/test/data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))

  input_dim: 576
  output_dim: 512


In [16]:
import sys
sys.path.append('/home/adelval/BTS/TFM/test/src/net')

from net_snr import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=False)
net_snr.load_theta('/home/adelval/BTS/TFM/test/data/model/theta_last')


  Net_snr:


    nb_params: 29.99M
    cuda: False
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/test/data/model/theta_last


In [17]:
# Model inference
import scipy.io
import matplotlib.pyplot as plt

net_snr.set_mode_train(False)

f = data_i['file']
#print(f)
x = data_i['x08k']
print(x[0,:5],x[0,512:517])
print(x[1,:5],x[1,512:517])
print(x[2,:5],x[2,512:517])
print(x[3,:5],x[3,512:517])
print(x[19,:5],x[19,512:517])

direc = os.path.dirname(f)
direc = direc.replace('data/audio', 'data/prueba', 1)
if not os.path.exists(direc):
    os.makedirs(direc)
out = f.replace('data/audio', 'data/prueba', 1)
out = out.replace('.wav', '.mat', 1)
    
print(' Checking: ' + out)
#if not os.path.exists(out):
print('Processing: ' + out)
snr = net_snr.predict(x)
print(snr[0,:])
snr = to_numpy(snr.squeeze())
print(snr.shape)
scipy.io.savemat(out, mdict={'snr': snr})
x = to_numpy(x.squeeze())
# snr = to_numpy(snr.squeeze())

print(snr[0,:10])
print(snr[1,:10])
print(snr[2,:10])
print(snr[3,:10])
print(snr[19,:10])



[-2.47386    -1.5200697  -1.0509288  -0.8417688  -0.70806146] [-2.984838  -2.8981009 -2.7997372 -2.7685244 -2.7185802]
[-1.5796448 -1.5199704 -1.7905464 -1.2009634 -0.7062188] [-2.9946365 -2.9120762 -2.796107  -2.7484264 -2.6457765]
[-1.3838824 -1.1336191 -1.7873496 -2.563182  -1.5647051] [-2.9946198 -2.90983   -2.8680854 -2.8481464 -2.7226143]
[-1.0911639  -1.4584094  -1.1114112  -0.90263665 -0.8511954 ] [-2.9289918 -2.861931  -2.8482666 -2.8872168 -2.740795 ]
[-1.3306018 -1.2445422 -1.3499471 -1.991468  -3.9184818] [-2.9293892 -2.771369  -2.8836453 -2.8812828 -2.8127995]
 Checking: /home/adelval/BTS/TFM/audios/audio_1.mat
Processing: /home/adelval/BTS/TFM/audios/audio_1.mat
torch.Size([1, 466, 576])
del bloque 1 torch.Size([1, 512, 466])
SGE prev 0 tensor([[[ 5.5688e-01,  5.4133e-01,  5.5474e-01,  ...,  5.8574e-01,
           7.7875e-01,  9.9029e-01],
         [-6.2923e-01, -6.1514e-01, -6.2426e-01,  ..., -1.9666e+00,
          -2.1871e+00, -2.4252e+00],
         [ 2.0554e-01,  1.974

EVALUATION

In [18]:
# Real evaluation

from __future__ import print_function
from __future__ import division
import time, os, sys
import warnings
warnings.simplefilter('ignore')

sys.path.append('/home/adelval/BTS/TFM/test/src/train')
sys.path.append('/home/adelval/BTS/TFM/test/src/eval')

import numpy as np
from eval_utils import *
from vvtk_net.config import Configuration
from scipy.io import wavfile
from scipy.io import loadmat

In [19]:
print(x_test)

['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [20]:
# Aplicar mascara al audio completo

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    it = int(np.floor((data.size-frame)/shift))
    print(f'La ventana se desplazara {it} veces')
    
    for i in range(0,it):
        print(f'{i} El frame sin enventanado con rn resulta {data[i*shift:i*shift+10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')

        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(Xfft.shape)
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')

        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')

        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        if i == 0:
            print(f'El primer frame mejorado es {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


fs=16000
B=[32]
w=[0.040]
frame = int(fs*w[0])
m=0.010
shift = int(fs*m)
nfft=1024
gmin = 0.0562

#wav = '/home/adelval/BTS/TFM/test/'+ x_test[0]
wav = x_test[0]
print(f'Audio seleccionado es --> {wav}')

fs, x = wavfile.read(wav) 
x = np.array(x, dtype=np.float32) / 2 ** 15
print(f'El frame sin enventanado resulta {x[:10]}')
snr_net_file = wav.replace('.wav','.mat').replace('audio','out') 
print(snr_net_file)
snr_net_file = '/home/adelval/BTS/TFM/audios/audio_1.mat'
snr_net = loadmat(snr_net_file)['snr'].T
print(f'La mascara del primer fragmento es {snr_net[:,0]}')
    
xenh, filt = noiseReduction(x, snr_net, fs, frame, shift, nfft, gmin)
print(f'Las dimensiones del filtro son {filt.shape}')
print(f'El frame mejorado es {xenh[:100]}')

wavenh = wav.replace('audio','prueba')    
if not os.path.isdir(os.path.dirname(wavenh)):
    os.makedirs(os.path.dirname(wavenh))
wavfile.write(wavenh,fs,xenh)
    
snr = int(wada_snr(xenh))
print('snr(wada)=%idB, file: %s' % (snr, wav))


Audio seleccionado es --> /home/adelval/BTS/TFM/audios/audio_1.wav
El frame sin enventanado resulta [ 0.0000000e+00 -3.0517578e-05 -3.0517578e-05  0.0000000e+00
 -3.0517578e-05  0.0000000e+00  0.0000000e+00  0.0000000e+00
  3.0517578e-05  3.0517578e-05]
/home/adelval/BTS/TFM/outs/out_1.mat
La mascara del primer fragmento es [0.33868918 0.37613457 0.3741048  0.34872591 0.36367503 0.40903547
 0.46936008 0.51463634 0.5507999  0.5669161  0.6032347  0.6130794
 0.63549405 0.6692364  0.68757457 0.71275425 0.749676   0.77683806
 0.79358053 0.8038157  0.7944289  0.8028796  0.79731554 0.7960488
 0.80437374 0.7968596  0.7954531  0.8066587  0.8023248  0.8062155
 0.8139721  0.8351213  0.8612058  0.87144166 0.87458897 0.8822209
 0.8831611  0.8865919  0.8987105  0.90113443 0.90174466 0.90874696
 0.9164089  0.92978346 0.9360894  0.94059396 0.94745183 0.95301783
 0.95748913 0.9608601  0.96132696 0.96468484 0.9626491  0.9606947
 0.9657568  0.9680273  0.97003    0.9699621  0.96952033 0.97039044
 0.971044

El tamaño de la salida del filtro sera (513,)
La salida del filtro es [ 0.00624033+0.j          0.00379512+0.00447065j -0.00536751-0.00540168j
  0.02812334+0.00516897j  0.00192906-0.02669484j -0.02284549-0.02190402j
  0.0021734 -0.00534672j -0.03185189+0.04979549j  0.06488861-0.0775569j
  0.03642182+0.02279475j]
El frame mejorado resulta sin OLA es [-0.0001431  -0.00044797 -0.00067199 -0.00025558  0.00052957  0.00065015
 -0.00027327 -0.00049518  0.00023596  0.00115635]
329 El frame sin enventanado con rn resulta [-0.03976434 -0.04034415 -0.00811762  0.03622446  0.04269419  0.00115971
 -0.01174923  0.01147464  0.03335576  0.04696662]
El frame enventanado resulta [-0.00000000e+00 -1.98348016e-04 -7.98179950e-05  5.34264669e-04
  8.39556237e-04  2.85051370e-05 -3.46534782e-04  3.94821132e-04
  1.31158919e-03  2.07749066e-03]
(513,)
El tamaño de la transformada del frame es (513,)
La transformada del frame es [ 0.01086016+0.j          0.00089817-0.00301183j  0.01337963+0.0033186j
 -0.00314